# 01 · Tensor & Autograd

PyTorch의 심장부 두 가지:
1. **Tensor** — numpy 배열과 비슷하지만 GPU에 얹고 자동 미분이 된다.
2. **Autograd** — 연산 그래프를 자동으로 추적하고 `.backward()` 한 번에 기울기 계산.

이 노트북 끝엔 **Linear Regression을 PyTorch로 처음부터** 만들어봅니다.

## 1. Tensor 기초 (numpy와 평행)

In [ ]:
import torch
import numpy as np

a = torch.tensor([[1., 2.], [3., 4.]])
b = torch.ones(2, 2)
print('a =', a)
print('b =', b)
print('a + b =', a + b)
print('a @ b =', a @ b)
print('a.shape =', a.shape, '| a.dtype =', a.dtype)

In [ ]:
# numpy ↔ torch 변환
arr = np.random.randn(3, 3)
t = torch.from_numpy(arr)
back = t.numpy()
print('numpy → torch → numpy 왕복 OK:', np.allclose(arr, back))

## 2. Autograd — 자동 미분

`requires_grad=True`를 준 tensor로 계산하면 **연산 그래프가 기록**된다.
스칼라 결과에 `.backward()`를 부르면 모든 leaf tensor의 `.grad`가 채워진다.

In [ ]:
# y = x^2 + 3x + 1 의 x=2에서 미분 → 2x + 3 = 7
x = torch.tensor(2.0, requires_grad=True)
y = x**2 + 3*x + 1
y.backward()
print(f'x = {x.item()}, y = {y.item()}')
print(f'dy/dx at x=2 = {x.grad.item()}  (기대값 7)')

## 3. 손으로 만드는 Linear Regression

`y = w*x + b`를 데이터에 맞추기. 목적은 **학습 루프를 이해**하는 것.

In [ ]:
# 가짜 데이터: y = 2x + 1 + noise
torch.manual_seed(0)
X = torch.linspace(-3, 3, 100).unsqueeze(1)   # shape (100, 1)
true_y = 2 * X + 1
y = true_y + 0.5 * torch.randn_like(true_y)

# 학습할 파라미터
w = torch.zeros(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
lr = 0.05

history = []
for step in range(100):
    # forward
    pred = w * X + b
    loss = ((pred - y)**2).mean()       # MSE

    # backward
    loss.backward()

    # update (gradient descent). no_grad로 감싸야 autograd가 이 step을 또 기록 안 함.
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad
        w.grad.zero_()
        b.grad.zero_()

    if step % 10 == 0:
        history.append((step, loss.item(), w.item(), b.item()))
        print(f'step {step:3d} | loss={loss.item():.3f} | w={w.item():.3f} b={b.item():.3f}')

print(f'\n최종: w={w.item():.3f} (target 2.0), b={b.item():.3f} (target 1.0)')

In [ ]:
import matplotlib.pyplot as plt
plt.scatter(X.numpy(), y.numpy(), alpha=0.5, label='data')
plt.plot(X.numpy(), (w*X + b).detach().numpy(), 'r-', label=f'y = {w.item():.2f}x + {b.item():.2f}')
plt.legend(); plt.title('Linear Regression from scratch')
plt.show()

## 4. 핵심 요약

방금 한 것이 딥러닝의 **전부**이다. 모델만 크게 갈아끼우면 된다:

```
for step:
    pred = model(X)       # forward
    loss = loss_fn(pred, y)
    loss.backward()       # backward (autograd)
    optimizer.step()      # update
    optimizer.zero_grad()
```

다음 노트북에서 `nn.Module`, `DataLoader`, `optim`으로 **실제 신경망 학습 루프**를 써봅니다.

### AI agent에 물어볼 것
1. "requires_grad=True를 안 준 tensor로 학습하면 어떻게 돼?"
2. "왜 `.grad.zero_()`를 매 step마다 해줘야 해? 안 하면 어떻게 돼?"
3. "Learning rate를 0.5로 높이면 어떤 현상이 나타날까? 직접 바꿔 실행해봐."